# Agent Loop from Scratch
Building a basic agent loop from first principles.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv(override=True)

In [ ]:
import os
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

### 1. Define a tool
Let's create a simple tool that the agent can use.

In [ ]:
def get_weather(location: str) -> str:
    """Get the current weather in a given location"""
    # In a real app, this would call an external API
    if "boston" in location.lower():
        return "It is currently 60 degrees and raining in Boston."
    elif "new york" in location.lower():
        return "It is currently 70 degrees and sunny in New York."
    return f"It is 65 degrees and cloudy in {location}."

get_weather_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA"
                }
            },
            "required": ["location"],
            "additionalProperties": False
        }
    }
}

tools = [get_weather_schema]

### 2. The Agent Loop
The loop will continue as long as the LLM decides to call tools (`finish_reason == "tool_calls"`).

In [ ]:
def run_agent_loop(messages, tools):
    print("Starting agent loop...")
    
    # 1. Initial LLM call
    response = openrouter.chat.completions.create(
        model="nemotron-3-super-120b-a12b:free",
        messages=messages,
        tools=tools
    )
    
    # 2. Loop while the agent wants to use tools
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        
        # 3. Execute all tool calls
        for tool_call in message.tool_calls:
            print(f"\n[Agent called tool: {tool_call.function.name}]")
            
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            # Dispatch to our python functions
            if tool_name == "get_weather":
                result = get_weather(**arguments)
            else:
                result = "Unknown tool"
                
            print(f"[Tool result: {result}]")
            
            # Add tool result to history
            messages.append({
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            })
            
        # 4. Send updated history back to LLM
        response = openrouter.chat.completions.create(
            model="nemotron-3-super-120b-a12b:free",
            messages=messages,
            tools=tools
        )
        
    # 5. Final conversational response
    print("\nFinal Response:")
    print(response.choices[0].message.content)
    return messages

### 3. Test the loop

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use your tools to answer the user's questions."},
    {"role": "user", "content": "What's the weather like in Bengaluru, New York and Moscow right now?"}
]

final_history = run_agent_loop(messages, tools)